In [2]:
import pytesseract, re
from unidecode import unidecode
from PIL import Image
from pathlib import Path
import spacy
from typing import List, Union, Dict
from gensim.models import Phrases
from gensim.models.phrases import Phraser
import json
import pandas as pd
from collections import defaultdict
import time
import ijson
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.sparse import csr_matrix
from typing import Tuple
import numpy as np
import seaborn as sns
from emoji import demojize
from transformers import AutoTokenizer
from tqdm import tqdm

c:\Users\Mateus Monteleone\Projects\ic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def extractTweetData(tweets):
    # Extract hashtag texts from entities.hashtags list
    hashtags = [tag['text'] for tag in tweets.get('entities', {}).get('hashtags', [])]

    # Clean the 'source' HTML anchor tag, extract just the platform name
    import re
    source_html = tweets.get('source', '')
    match = re.search(r'>(.*?)<', source_html)
    source_clean = match.group(1) if match else source_html

    return {
        'user_id': tweets['user']['id_str'],
        'screen_name': tweets['user']['screen_name'],
        'name': tweets['user']['name'],
        'location': tweets['user'].get('location', None),
        'followers_count': tweets['user'].get('followers_count', 0),
        'created_at': tweets['created_at'],
        'full_text': tweets['full_text'],
        'retweet_count': tweets.get('retweet_count', 0),
        'favorite_count': tweets.get('favorite_count', 0),
        'lang': tweets.get('lang', None),
        'hashtags': hashtags,
        'source': source_clean,
    }

In [4]:
def streamTweetsFromFile(file_path, max_tweets=None, log_every=5000):
    tweets = []
    tweet_count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        # ijson parses multiple JSON objects from the stream
        parser = ijson.items(f, '', multiple_values=True)

        for tweet in parser:
            tweets.append(tweet)
            tweet_count += 1

            if tweet_count % log_every == 0:
                print(f"[{tweet_count}] tweets loaded...")

            if max_tweets and tweet_count >= max_tweets:
                break

    print(f"✅ Done. Loaded {tweet_count} tweets.")
    return tweets



In [ ]:
tweets = streamTweetsFromFile(r"C:\Users\Mateus Monteleone\Projects\ic\data\tweets.jsonl", log_every=5000)

[5000] tweets loaded...
[10000] tweets loaded...
[15000] tweets loaded...
[20000] tweets loaded...
[25000] tweets loaded...
[30000] tweets loaded...
[35000] tweets loaded...
[40000] tweets loaded...
[45000] tweets loaded...
[50000] tweets loaded...
[55000] tweets loaded...
[60000] tweets loaded...
[65000] tweets loaded...
[70000] tweets loaded...
[75000] tweets loaded...
[80000] tweets loaded...
[85000] tweets loaded...
[90000] tweets loaded...
[95000] tweets loaded...
[100000] tweets loaded...
[105000] tweets loaded...
[110000] tweets loaded...
[115000] tweets loaded...
[120000] tweets loaded...
[125000] tweets loaded...
[130000] tweets loaded...
[135000] tweets loaded...
[140000] tweets loaded...
[145000] tweets loaded...
[150000] tweets loaded...
[155000] tweets loaded...
[160000] tweets loaded...
[165000] tweets loaded...
[170000] tweets loaded...
[175000] tweets loaded...
[180000] tweets loaded...
[185000] tweets loaded...
[190000] tweets loaded...
[195000] tweets loaded...
[20000

In [7]:
len(tweets)

270301

In [9]:
print(tweets[0])

{'created_at': 'Sat Oct 01 23:59:59 +0000 2022', 'id': 1576361316239175680, 'id_str': '1576361316239175680', 'full_text': 'se o lula ganhar eu quero uma roda de beijo com 13 bofes e o primeiro vai beijando os 13 até chegar no segundo que inicia o ciclo novamente e por aí vai', 'truncated': False, 'display_text_range': [0, 152], 'entities': {'hashtags': [], 'symbols': [], 'user_mentions': [], 'urls': []}, 'metadata': {'iso_language_code': 'pt', 'result_type': 'recent'}, 'source': '<a href="http://twitter.com/download/iphone" rel="nofollow">Twitter for iPhone</a>', 'in_reply_to_status_id': None, 'in_reply_to_status_id_str': None, 'in_reply_to_user_id': None, 'in_reply_to_user_id_str': None, 'in_reply_to_screen_name': None, 'user': {'id': 772193576, 'id_str': '772193576', 'name': 'paulo rouge', 'screen_name': 'pauloisfab', 'location': 'Sao Paulo, Brazil', 'description': 'amor e paixão, viados no meu coração', 'url': 'https://t.co/Cusc3NHqAV', 'entities': {'url': {'urls': [{'url': 'https:/

In [ ]:
df = pd.DataFrame([extractTweetData(t) for t in tweets])

In [11]:
len(df['full_text'])

270301

In [12]:
df.head(4)

,user_id,screen_name,name,location,followers_count,created_at,full_text,retweet_count,favorite_count,lang,hashtags,source
0,772193576,pauloisfab,paulo rouge,"Sao Paulo, Brazil",439,Sat Oct 01 23:59:59 +0000 2022,se o lula ganhar eu quero uma roda de beijo co...,0,0,pt,[],Twitter for iPhone
1,1546155973051662336,jorgehen_pr,quem me conhece sabe 🚩,,27,Sat Oct 01 23:59:59 +0000 2022,@ixslorena LULA PRESIDENTE HOJE,0,0,pt,[],Twitter for Android
2,1515570566736003072,aanandaaluz,Ananda,pqp,14,Sat Oct 01 23:59:59 +0000 2022,mãe morrendo de alegria na carreata do Lula,0,3,pt,[],Twitter for Android
3,1449859889589899264,lyeroses,sasa៹ 13 ⭐🚩,only blink,4973,Sat Oct 01 23:59:59 +0000 2022,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,0,3,pt,[],Twitter for Android


In [ ]:
df.to_csv(r"C:\Users\Mateus Monteleone\Projects\ic\data\dados_gerais_bronze1.csv")

In [ ]:
def extractRawTweets(df: pd.DataFrame) -> pd.Series:
    """
    Extracts the 'full_text' column from the DataFrame.

    This extracted data (the raw tweets) is the input for the 
    Pre-Processing phase, where stopwords are removed and stemming 
    is applied, as described in the methodology [2, 3].
    """
    
    # Check if the column exists to prevent errors
    if 'full_text' in df.columns:
        # Extract the column. We return a pandas Series of strings.
        tweets_series = df['full_text']
        print(f"Successfully extracted {len(tweets_series)} raw texts.")
        return tweets_series
    else:
        print("Error: The DataFrame does not contain a column named 'full_text'.")
        return pd.Series(dtype=object)

# Example usage (assuming 'df' is loaded):
# raw_texts = extract_raw_tweets(df)

In [ ]:
extractRawTweets(df).to_csv(r"C:\Users\Mateus Monteleone\Projects\ic\data\tweets_brutos_bronze2.csv")